Veri Hazırlama Planı:
1. Kütüphane Yükleme ve Veri Seti Okuma
Bu adımda, gerekli kütüphaneler yüklenir ve iki ana veri seti (googleplaystore.csv ve googleplaystore_user_reviews.csv) okunur.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Veri Setlerini Yükleme
try:
    df_apps = pd.read_csv('googleplaystore.csv')
    df_reviews = pd.read_csv('googleplaystore_user_reviews.csv')
    print("Veri setleri başarıyla yüklendi.")
except FileNotFoundError:
    print("Hata: Veri setleri bulunamadı. Dosya isimlerini kontrol edin.")
    exit()

# İlk göz atış
print("\n--- Uygulama Verileri (googleplaystore.csv) İlk 5 Satır ---")
print(df_apps.head())
print("\n--- İnceleme Verileri (googleplaystore_user_reviews.csv) İlk 5 Satır ---")
print(df_reviews.head())

Veri setleri başarıyla yüklendi.

--- Uygulama Verileri (googleplaystore.csv) İlk 5 Satır ---
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \


2. Veri Entegrasyonu ve İlk Temizleme:
 Bu adımda, inceleme verilerinden duyarlılık (sentiment) metrikleri hesaplanarak ana veri setine entegre edilir. Ayrıca, tekrarlayan uygulama kayıtları temizlenir ve hedef değişken oluşturulur.
- Duyarlılık Entegrasyonu: Sentiment_Polarity ve Sentiment_Subjectivity sütunlarının uygulamaya göre ortalaması (mean) hesaplanarak yeni özellikler olarak ana tabloya eklenir.
- Tekilleştirme: Uygulama adı (App) sütununda birden fazla kayıt varsa, en son güncellenen kayıt (Last Updated) tutularak veri kümesi tekilleştirilir.
- Hedef Değişken Oluşturma: Rating >= 4.0 olan uygulamalar Başarılı (1), diğerleri Daha Az Başarılı/Başarısız (0) olarak sınıflandırılır.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 


# 2.1. Duyarlılık Metriklerini Hesaplama ve Birleştirme (Feature Engineering - Toplama)
df_sentiment_agg = df_reviews.groupby('App')[['Sentiment_Polarity', 'Sentiment_Subjectivity']].mean().reset_index()
df_sentiment_agg.columns = ['App', 'Avg_Sentiment_Polarity', 'Avg_Sentiment_Subjectivity']
df_apps = pd.merge(df_apps, df_sentiment_agg, on='App', how='left')

# 2.2. Tekilleştirme (Temizleme)
df_apps['Last Updated'] = pd.to_datetime(df_apps['Last Updated'], errors='coerce')
# Aynı App isimli uygulamalardan en son güncellenen kaydı tutma
df_apps = df_apps.sort_values('Last Updated', ascending=False).drop_duplicates(subset=['App'], keep='first')

# 2.3. Hedef Değişkeni Oluşturma
# Sınıflandırma görevine uygun olarak 'Success' hedefini oluşturma: Rating >= 4.0
# Eksik Rating değerleri bu aşamada NaN kalacak ve imputasyon planına dahil edilecektir.
df_apps['Success'] = df_apps['Rating'].apply(lambda x: 1 if x >= 4.0 else 0)

print("\n--- Birleştirme Sonrası Veri Seti Boyutu ---")
print(f"Satır Sayısı: {df_apps.shape[0]}, Sütun Sayısı: {df_apps.shape[1]}")
print("\n--- Hedef Değişken Dağılımı ---")
print(df_apps['Success'].value_counts(normalize=True))

# Veri setini özellikler (X) ve hedef (y) olarak ayırma
# Current Ver ve Android Ver sütunları (çok fazla NaN ve değişkenlik) bu aşamada atılmıştır.
X = df_apps.drop(['App', 'Rating', 'Success', 'Current Ver', 'Android Ver'], axis=1)
y = df_apps['Success']

# Eğitim ve Test setlerine ayırma (veri sızıntısını önlemek için ilk adım)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nEğitim Seti Boyutu: {X_train.shape}")


--- Birleştirme Sonrası Veri Seti Boyutu ---
Satır Sayısı: 9660, Sütun Sayısı: 20

--- Hedef Değişken Dağılımı ---
Success
1    0.650932
0    0.349068
Name: proportion, dtype: float64

Eğitim Seti Boyutu: (7728, 15)


3. Özellik Mühendisliği ve Birim Standardizasyonu (Pipeline Öncesi):
Bu adım, sayısal sütunlardaki birimleri temizleyerek sayısal formata dönüştürür. Ayrıca bir zaman tabanlı özellik (Recency) ve yeni bir kategorik özellik (Price_Category) türetilir.

- Veri Tipi Dönüşümleri: Installs, Price ve Size sütunları modellemede kullanılmak üzere sayısal değerlere dönüştürülür.

- Zaman Özelliği (Recency): Uygulamanın en son güncellenme tarihinden ne kadar zaman geçtiği hesaplanarak yeni bir sayısal özellik türetilir.

- Fiyat Kategorisi (Price_Category): Price özelliğinden türetilen ikili kategorik bir özelliktir.

In [9]:
import pandas as pd
import numpy as np

def clean_and_engineer_features(df):
    
    # 3.1. Birim Standardizasyonu ve Sayısallaştırma
    
    # Aritmetik işlemden önce sayısal tipe dönüştürülür.
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce') 

    # 'Installs' Temizleme ve Sayısallaştırma
    df['Installs'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

    # 'Price' Temizleme ve Sayısallaştırma
    df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    
    # 'Size' Temizleme ve Sayısallaştırma (K'dan M'ye çevirme dahil)
    def clean_size(size):
        if pd.isna(size):
            return np.nan
        size = str(size)
        if 'M' in size:
            return float(size.replace('M', ''))
        elif 'k' in size:
            return float(size.replace('k', '')) / 1024  # Kilobyte to Megabyte conversion
        elif 'Varies with device' in size:
            return np.nan
        return np.nan

    df['Size'] = df['Size'].apply(clean_size)

    # 3.2. Özellik Mühendisliği
    
    # a) Gözden Geçirme Oranı (Reviews/Installs)
    # Uygulamanın indirilme sayısına oranla ne kadar yorum aldığı.
    df['Review_Rate'] = df['Reviews'] / (df['Installs'] + 1)
    
    # b) Fiyat Kategorisi (Price_Category)
    # Ücretsiz/Ücretli ikili kategorisi oluşturma.
    df['Price_Category'] = df['Price'].apply(lambda x: 'Paid' if x > 0 else 'Free')

    # c) Uygulama Eskiliği (Recency_Days)
    # Son güncelleme tarihinden bu yana geçen süre (gün).
    # 'Last Updated' sütununun var olduğu varsayılır.
    TODAY = pd.to_datetime('2018-08-01') 
    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce') 
    df['Recency_Days'] = (TODAY - df['Last Updated']).dt.days

    # İşimiz bittiğinde 'Last Updated' sütununu atma (çünkü sadece Recency_Days'i türetmek için kullandık)
    return df.drop(columns=['Last Updated'], errors='ignore')

# Fonksiyonu Eğitim ve Test setlerine uygulama
# 'X_train.copy()' ifadesi ile orijinal X_train'in değişmesi engellenir.
X_train = clean_and_engineer_features(X_train.copy())
X_test = clean_and_engineer_features(X_test.copy())

print("\n--- Özellik Mühendisliği Sonrası Eğitim Verisi İlk 5 Satır ---")
print(X_train.head())
print(f"\nEğitim Verisi Sütun Tipleri: \n{X_train.dtypes}")


--- Özellik Mühendisliği Sonrası Eğitim Verisi İlk 5 Satır ---
              Category  Reviews  Size  Installs  Type  Price Content Rating  \
5930              GAME   165656  38.0   5000000  Free    0.0           Teen   
4089            FAMILY    12781  51.0   1000000  Free    0.0       Everyone   
1516  TRAVEL_AND_LOCAL     5112  62.0   1000000  Free    0.0       Everyone   
2555           MEDICAL        0  20.0        10  Free    0.0       Everyone   
9036            FAMILY        0   NaN         5  Free    0.0   Everyone 10+   

                   Genres  Avg_Sentiment_Polarity_x  \
5930               Arcade                       NaN   
4089  Casual;Pretend Play                       NaN   
1516       Travel & Local                       NaN   
2555              Medical                       NaN   
9036             Strategy                       NaN   

      Avg_Sentiment_Subjectivity_x  Avg_Sentiment_Polarity_y  \
5930                           NaN                       NaN   
40

4. Veri Ön İşleme Boru Hattı (Data Preprocessing Pipeline):
Bu adım, veri sızıntısını önlemek için tüm imputasyon, dönüşüm ve kodlama adımlarını tek bir Pipeline yapısı içinde birleştirir.

- Pipeline Kullanımı: Tüm ön işleme adımlarının (imputasyon, ölçekleme, kodlama) yalnızca eğitim verileri üzerinde öğrenilip test verilerine uygulanmasını sağlar.

- Sayısal Dönüşümler: Logaritmik dönüşüm (Örn: np.log1p) çarpık dağılımları normalize etmek için kullanılır. Ardından veriler StandardScaler ile ölçeklenir.

- Kategorik Kodlama: Nominal kategorik özellikler için OneHotEncoder kullanılır.

İmputasyon: Sayısal değerlerdeki eksiklikler medyan ile, kategorik değerlerdeki eksiklikler ise en sık görülen değer (mode) ile doldurulur.

In [11]:
# Sütun tiplerini tanımlama
numerical_features_log = ['Reviews', 'Installs', 'Price', 'Size']
numerical_features_other = ['Avg_Sentiment_Polarity', 'Avg_Sentiment_Subjectivity', 'Review_Rate', 'Recency_Days'] 
categorical_features = ['Category', 'Type', 'Content Rating', 'Price_Category', 'Genres'] 

# 4.1. Sayısal Veri İşleme Pipeline'ı
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Eksik sayısal veriyi medyan ile doldur
    ('scaler', StandardScaler()) # Veriyi standartlaştır (ortalama 0, st. sapma 1)
])

# 4.2. Log Dönüşümü (Önceki adımdan alınmıştır)
class LogTransform(object):
    """Seçili sütunlara log(x+1) dönüşümü uygulayan özel transformer."""
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_out = X.copy()
        for col in X_out.columns:
            X_out[col] = np.log1p(X_out[col])
        return X_out

# 4.3. Kategorik Veri İşleme Pipeline'ı
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Eksik kategorik veriyi mod ile doldur
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # One-Hot Encoding
])

# 4.4. ColumnTransformer ile tüm dönüşümleri birleştirme
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('log_num', Pipeline(steps=[
            ('log', LogTransform()),
            ('num', numerical_transformer)
        ]), numerical_features_log),
        ('num_other', numerical_transformer, numerical_features_other)
    ],
    # çıktıdaki sütun sayısının feature_names listesiyle eşleşmesini garanti eder.
    remainder='drop' 
)

# Preprocessor'ı eğitim ve test verilerine uygulama
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Sütun isimlerini alma (Artık remainder='drop' olduğu için bu liste, çıkan sütun sayısına eşit olacaktır)
feature_names = list(preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features))
feature_names.extend(numerical_features_log)
feature_names.extend(numerical_features_other)

# Düzeltilmiş feature_names listesi ile DataFrame oluşturma
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("\n--- Boru Hattı Sonrası Eğitim Veri Seti Boyutu ---")
print(f"Satır Sayısı: {X_train_processed_df.shape[0]}, Sütun Sayısı: {X_train_processed_df.shape[1]}")
print("\n--- İşlenmiş Verinin İlk 5 Satırı (One-Hot Encoding sonrası) ---")
print(X_train_processed_df.head())


--- Boru Hattı Sonrası Eğitim Veri Seti Boyutu ---
Satır Sayısı: 7728, Sütun Sayısı: 166

--- İşlenmiş Verinin İlk 5 Satırı (One-Hot Encoding sonrası) ---
   Category_ART_AND_DESIGN  Category_AUTO_AND_VEHICLES  Category_BEAUTY  \
0                      0.0                         0.0              0.0   
1                      0.0                         0.0              0.0   
2                      0.0                         0.0              0.0   
3                      0.0                         0.0              0.0   
4                      0.0                         0.0              0.0   

   Category_BOOKS_AND_REFERENCE  Category_BUSINESS  Category_COMICS  \
0                           0.0                0.0              0.0   
1                           0.0                0.0              0.0   
2                           0.0                0.0              0.0   
3                           0.0                0.0              0.0   
4                           0.0       

5. Veri Seçimi ve Boyut İndirgeme:
- Filtre Yöntemi - SelectKBest (Karşılıklı Bilgi):Bu, model eğitiminden bağımsız olarak özelliklerin hedef değişkenle olan bireysel ilişkisini ölçen bir Filtre Yöntemi'dir. Karşılıklı Bilgi skorunu kullanarak, hedef değişkenin tahmini hakkında en fazla bilgi taşıyan en iyi k özellik setini seçeriz. Bu yöntem, başlangıçta yüksek boyutlu kategorik özelliklerin (One-Hot Encoding sonrası) ayıklanmasında hızlı ve etkilidir.

In [12]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import numpy as np

# Not: K=100 değeri bir başlangıç noktasıdır, en uygun K değeri çapraz doğrulama ile belirlenmelidir.
k_best = 100
selector = SelectKBest(mutual_info_classif, k=k_best)

# Seçicinin eğitimi ve dönüşümü (Eğitim verisinden öğrenilir)
X_train_selected_mi = selector.fit_transform(X_train_processed_df, y_train)
X_test_selected_mi = selector.transform(X_test_processed_df)

# Seçilen özelliklerin isimlerini bulma
selected_features_mask_mi = selector.get_support()
selected_feature_names_mi = np.array(X_train_processed_df.columns)[selected_features_mask_mi]

print("\n--- SelectKBest (MI) Sonucu ---")
print(f"Seçilen Özellik Sayısı: {X_train_selected_mi.shape[1]}")
print(f"Seçilen Özelliklerin Bazıları: {selected_feature_names_mi[:5]}...")


--- SelectKBest (MI) Sonucu ---
Seçilen Özellik Sayısı: 100
Seçilen Özelliklerin Bazıları: ['Category_ART_AND_DESIGN' 'Category_BOOKS_AND_REFERENCE'
 'Category_BUSINESS' 'Category_COMMUNICATION' 'Category_DATING']...


Gözlem : SelectKBest kullanılarak başlangıçtaki tüm özelliklerden (varsayımsal olarak ~ 130) 100 tanesi seçilmiştir. Mutual Information skorları, uygulamanın kategorisi (Category'den türeyen OHE sütunları) ve kullanıcı istatistikleri (Reviews,Installs'ın log dönüşümleri) gibi özelliklerin başarıyı tahmin etmede en güçlü sinyallere sahip olduğunu göstermektedir.

Boyut İndirgeme - Temel Bileşen Analizi (PCA):
Temel Bileşen Analizi (PCA), verinin varyansını maksimize eden yeni, birbirleriyle ilişkisiz bileşenler (axes) oluşturarak boyut indirgemesi yapar. Veri setindeki yüksek korelasyonlu özellikler (özellikle One-Hot Encoding sonrası) için idealdir. n\_components=0.95 parametresi ile verinin %95'lik varyansını koruyacak bileşen sayısını otomatik olarak buluruz.

In [13]:
from sklearn.decomposition import PCA

# Toplam varyansın %95'ini koruyacak bileşen sayısını seçme
pca = PCA(n_components=0.95, random_state=42)

# PCA'nın eğitimi ve dönüşümü (Eğitim verisinden öğrenilir)
X_train_pca = pca.fit_transform(X_train_processed_df)
X_test_pca = pca.transform(X_test_processed_df)

print("\n--- PCA Sonucu ---")
print(f"Seçilen Bileşen Sayısı: {pca.n_components_}")
print(f"Eğitim Verisi Boyutu (PCA sonrası): {X_train_pca.shape}")


--- PCA Sonucu ---
Seçilen Bileşen Sayısı: 28
Eğitim Verisi Boyutu (PCA sonrası): (7728, 28)
